# Interview Preparation QA - AI Engineer

QA for AI Engineer Interview Preparation using Vectorless RAG approach (pageindex)

## Working of Vectorless RAG

Traditional RAG → chunk → embed → cosine similarity → retrieve
PageIndex RAG → build tree → LLM reasons over tree → retrieve exact sections

- Will write more in detail later

### 1. Installation and Setup

In [12]:
# Installing necessary packages
!pip  install -U pageindex python-dotenv groq


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
# Create a .env file
# Uncomment and fill in your key, then run this cell ONCE

# env_content = """
# PAGEINDEX_API_KEY=your_pageindex_key_here
# """

# with open(".env", "w") as f:
#     f.write(env_content.strip())
# print(".env file created")

.env file created


In [13]:
import os,json
from dotenv import load_dotenv

load_dotenv()

PAGEINDEX_API_KEY=os.getenv("PAGEINDEX_API_KEY")
GROQ_API_KEY=os.getenv("GROQ_API_KEY")

print("PageIndex key loaded:", "✅" if PAGEINDEX_API_KEY else "❌ Missing!")
print("GROQ API key loaded:", "✅" if GROQ_API_KEY else "❌ Missing!")

PageIndex key loaded: ✅
GROQ API key loaded: ✅


In [14]:
from pageindex import PageIndexClient
from groq import Groq

pi_client=PageIndexClient(api_key=PAGEINDEX_API_KEY)
groq_client=Groq(api_key=GROQ_API_KEY)

print("✅ PageIndex client ready")
print("✅ GROQ client ready")

✅ PageIndex client ready
✅ GROQ client ready


### 2. Upload and Index a PDF

- Upload your PDF to the PageIndex cloud
- PageIndex uses an LLM to read the document structure
- Builds a hierarchical tree index (like a smart Table of Contents)
- Returns a doc_id for all future operations

In [ ]:
PDF_PATH='data/genai-interview-questions.pdf'
doc_id = None

# Check if .env exists and contains DOCUMENT_ID
# Done to prevent file reupload -> costs credits oon PageIndex
if os.path.exists(".env"):
    with open(".env", "r") as f:
        content = f.read().strip()
        if "DOCUMENT_ID=" in content:
            doc_id = content.split("DOCUMENT_ID=")[-1].strip()
            print(f"Document already available\nDocument ID: {doc_id}")

# If DOCUMENT_ID not found, upload file
if not doc_id:
    print(f" Uploading file from {PDF_PATH} to PageIndex")
    result=pi_client.submit_document(file_path=PDF_PATH)
    doc_id=result['doc_id']

    print(f"Uploaded!")
    print(f"Document ID: {doc_id}")

    # Saving document id in env
    env_content= f"DOCUMENT_ID={doc_id}"

    # Save document ID to .env
    with open(".env", "a") as f:
        f.write(f"\nDOCUMENT_ID={doc_id}")

Document already available
Document ID: pi-cmnrbk7ua008101qws1uqyn5n


In [7]:
# PageIndex builds the tree asynchronously.
print("Building tree index...")
print("This runs once per document — the index is cached for reuse")

while True:
    status_result=pi_client.get_document(doc_id)
    status=status_result.get('status')
    print(f"Status: {status}")

    if status=="completed":
        print("\nTree index ready!")
        break
    elif status == "failed":
        print("\nProcessing failed. Check your PDF format.")
        break

Building tree index...
This runs once per document — the index is cached for reuse
Status: completed

Tree index ready!


### 3. Tree Structure

Each node has:
- node_id — unique ID used during retrieval
- title — section heading
- page_index — page number in original PDF
- text — section summary (when node_summary=True)
- nodes — child sections (nested)
- This structure is what the LLM reasons over during retrieval.

In [8]:
# fetch tree structure
tree_structure=pi_client.get_tree(doc_id, node_summary=True)

pageindex_tree=tree_structure.get('result',[])

print(f"Top-level sections: {len(pageindex_tree)}")
print("\nRaw tree (first node):")
print(json.dumps(pageindex_tree[0] if pageindex_tree else {}, indent=2))

Top-level sections: 1

Raw tree (first node):
{
  "title": "Generative AI",
  "node_id": "0000",
  "page_index": 1,
  "prefix_summary": "# Generative AI\n",
  "text": "# Generative AI\n",
  "nodes": [
    {
      "title": "Interview Q &amp; A",
      "node_id": "0001",
      "page_index": 1,
      "prefix_summary": "This document is an 'Interview Q & A' resource, featuring a 'GenAI Roadmap' and several images. It provides a detailed Table of Contents covering a wide range of topics related to Generative and Discriminative AI, including Transformer Architecture, LLMs, Embedding Models, RAG, LLM Fine Tuning, and various Prompt Engineering techniques (One-Shot, Few-Shot, Zero-Shot, Chain of Thought, Hybrid, ReAct). It also delves into advanced topics like GraphRAG and AI Agents, presents recent Q&A from company interviews, LLM leaderboards (including LMSYS), and modern LLM trends for 2024-2026. Finally, it includes a compilation of research papers from 2017 to 2025.",
      "text": "## In

In [9]:
# Pretty-print the full tree
def print_tree(nodes, indent=0):
    """Recursively print tree titles for a visual overview."""
    for node in nodes:
        prefix = "  " * indent + ("└─ " if indent > 0 else "")
        page   = node.get("page_index", "?")
        print(f"{prefix}[{node['node_id']}] {node['title']}  (p.{page})")
        if node.get("nodes"):
            print_tree(node["nodes"], indent + 1)

print("Full Document Structure:\n")
print_tree(pageindex_tree)

Full Document Structure:

[0000] Generative AI  (p.1)
  └─ [0001] Interview Q &amp; A  (p.1)
    └─ [0002] Generic Questions  (p.3)
    └─ [0003] Generative AI  (p.9)
    └─ [0004] Discriminative AI  (p.11)
    └─ [0005] General Questions on Generative and Discriminative AI  (p.12)
    └─ [0006] Transformer Architecture  (p.12)
    └─ [0007] Large Language Models (LLMs)  (p.17)
    └─ [0008] Retrieval-Augmented Generation (RAG)  (p.22)
    └─ [0009] LLM Fine Tuning  (p.24)
    └─ [0010] Misc Topics in Generative AI  (p.30)
    └─ [0011] Prompt Engineering  (p.31)
      └─ [0012] Q: What is adaptive prompting?  (p.34)
      └─ [0013] Q: How is prompt engineering used in interactive applications like chatbots?  (p.34)
      └─ [0014] Q: Provide a case study example of an effective prompt for content generation.  (p.34)
      └─ [0015] Q: Why is clarity important when designing prompts?  (p.34)
      └─ [0016] Q: How do you tailor prompts to the complexity of the task?  (p.34)
      └─ [0

### 4. LLM Tree Search

PageIndex retrieval:
query + tree → LLM reasons → "node 0007 and 0008 contain the answer"
Advantage: LLM understands document structure, context, and intent

The LLM acts like a human expert scanning a Table of Contents.

In [16]:
# Compress tree to save tokens — only send titles + short summaries
def compress_tree(nodes):
    output=[]

    for node in nodes:
        entry = {
            "node_id": node['node_id'],
            "title": node['title'],
            "page": node.get('page_index', '?'),
            "summary": node.get('text', '')[:150] # Trim to the first n (150) characters
        }

        # Checking for child nodes
        if node.get('nodes'):
            entry['children'] = compress_tree(node['nodes'])

        output.append(entry)
    
    return output

In [21]:
# Setting the prompt for LLM Tree Search
NODE_RETRIEVAL_PROMPT = """You are given a query and a document's tree structure (like a Table of Contents).
Your task: identify which node IDs most likely contain the answer to the query.
Think step-by-step about which sections are relevant.

Query: {query}

Document Tree:
{tree}

Reply ONLY in this exact JSON format:
{{
  "thinking": "<your step-by-step reasoning>",
  "node_list": ["node_id1", "node_id2"]
}}
"""

In [ ]:
def llm_tree_search(query: str, tree: list, model: str = 'openai/gpt-oss-120b'):
    # compressing the tree initially
    compressed_tree=compress_tree(tree)
    tree_str = json.dumps(compressed_tree, indent=2)

    response = groq_client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": "You are a precise JSON generator."},
            {"role": "user", "content": NODE_RETRIEVAL_PROMPT.format(query=query,tree=tree_str)}
        ],
        temperature=0,  # IMPORTANT for structured output
    )
    
    return json.loads(response.choices[0].message.content)

In [ ]:
# Testing the Retrieval
query = "Explain how does a Generative Adversarial Network (GAN) work?"

print(f"Query: {query}\n")
result = llm_tree_search(query, pageindex_tree)

print("LLM Reasoning:")
print(result.get("thinking", "N/A"))
print()
print("Selected Node IDs:", result.get("node_list", []))

🔍 Query: Explain how does a Generative Adversarial Network (GAN) work?

Model Response: 
{
  "thinking": "The query asks for an explanation of how a Generative Adversarial Network (GAN) works. The document tree is organized around generative AI topics. The only nodes that explicitly discuss generative models are the 'Generative AI' node (0003) and the broader 'General Questions on Generative and Discriminative AI' node (0005). While neither summary mentions GANs directly, these sections are the most relevant places where a GAN explanation would likely be found. No other node titles or summaries indicate coverage of GANs. Therefore, the answer is most likely located in nodes 0003 and 0005.",
  "node_list": ["0003", "0005"]
}
🧠 LLM Reasoning:
The query asks for an explanation of how a Generative Adversarial Network (GAN) works. The document tree is organized around generative AI topics. The only nodes that explicitly discuss generative models are the 'Generative AI' node (0003) and the b

### 5. Full End-to-End Pipeline

3 steps:

- Tree Search → LLM picks relevant node_ids
- Retrieve → Fetch the actual section content from those nodes
- Generate → LLM writes a grounded answer with page citations

What makes this better than vector RAG:

Retrieved content has titles + page numbers (traceable)
LLM can cite exactly which section the answer comes from
No hallucination from irrelevant chunks

In [35]:
# Helper function: Find nodes by ID 
def find_nodes_by_ids(tree:list, target_ids: list):
    found = []
    for node in tree:
        if node["node_id"] in target_ids:
            found.append(node)
        
        # Check and parse child nodes too
        if node.get('nodes'):
            found.extend(find_nodes_by_ids(node["nodes"], target_ids))

    return found

In [29]:
GENERATION_PROMPT = """You are an expert document analyst.
Answer the question using ONLY the provided context.
For every claim you make, cite the section title and page number in parentheses.
Be concise and precise.

Question: {query}

Context:
{context}

Answer:
"""

In [ ]:
def generate_answer(query: str, nodes: list, model: str = "openai/gpt-oss-120b") -> str:
    """
    Takes retrieved nodes as context and generates a grounded answer.
    Instructs the LLM to cite section titles and page numbers.
    """
    if not nodes:
        return "No relevant sections found in the document."
    
    # Build context string from retrieved nodes
    context_parts = []
    for node in nodes:
        context_parts.append(
            f"[Section: '{node['title']}' | Page {node.get('page_index', '?')}]\n"
            f"{node.get('text', 'Content not available.')}"
        )
    
    context = "\n\n---\n\n".join(context_parts)

    response = groq_client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": "You strictly answer using provided context and include citations."
            },
            {
                "role": "user",
                "content": GENERATION_PROMPT.format(query=query, context=context)
            }
        ],
        temperature=0.2,
    )

    return response.choices[0].message.content

In [39]:
# Vectorless RAG Pipeling
def vectorless_rag_pipeline(query: str, tree: list, verbose: bool = True) -> str:
    """
    Full end-to-end PageIndex RAG pipeline:
    
    Step 1: LLM Tree Search  → finds relevant node_ids
    Step 2: Node Retrieval   → fetches section content
    Step 3: Answer Generation → produces cited answer
    """
    if verbose:
        print(f"{'='*55}")
        print(f"Query: {query}")
        print(f"{'='*55}")
    
    # Step 1: Tree Search
    search_result = llm_tree_search(query, tree)
    node_ids = search_result.get("node_list", [])
    
    if verbose:
        print(f"\nReasoning: {search_result.get('thinking', '')[:200]}...")
        print(f"Retrieved node IDs: {node_ids}")
    
    # Step 2: Retrieve nodes
    nodes = find_nodes_by_ids(tree, node_ids)
    
    if verbose:
        print(f"Sections found: {[n['title'] for n in nodes]}")
    
    # Step 3: Generate answer
    answer = generate_answer(query, nodes)
    
    if verbose:
        print(f"\nAnswer:\n{answer}")
    
    return answer

In [40]:
answer = vectorless_rag_pipeline(
    query="Explain transformer architecture?",
    tree=pageindex_tree
)

Query: Explain transformer architecture?
Model Response: 
{
  "thinking": "The query asks for an explanation of the transformer architecture. Scanning the document tree, the node with title 'Transformer Architecture' (node_id 0006) directly addresses this topic, as indicated by its summary and title. No other nodes appear to focus on transformers, so the most relevant node is 0006.",
  "node_list": ["0006"]
}

Reasoning: The query asks for an explanation of the transformer architecture. Scanning the document tree, the node with title 'Transformer Architecture' (node_id 0006) directly addresses this topic, as indicated...
Retrieved node IDs: ['0006']
Sections found: ['Transformer Architecture']

Answer:
The Transformer is a deep‑learning model built around **self‑attention** rather than recurrence. Its core consists of two parallel stacks – an **encoder** and a **decoder** – each made of repeated layers that contain:

* **Self‑attention (or masked self‑attention in the decoder)** – comp